# Length-bias analysis

This notebook checks whether the local AI-text classifiers behave differently for short and long documents. It evaluates logistic regression, DistilBERT, DistilBERT with LoRA and MiCA, ModernBERT, and the GPT-2 and Qwen3 readout variants on the held-out test split.

Length effects require some care. A model may become more confident as it sees more evidence, which is useful rather than biased. The more concerning pattern is a same-direction score shift for both human and AI text, or a sharp change in false-positive or false-negative rates. Source and generator distributions also vary with length, so this is a diagnostic analysis rather than a causal claim.

In [ ]:
import gc
import hashlib
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
def find_project_dir(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the project directory containing pyproject.toml"
    )


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
from ai_detector import (
    artifact_status,
    load_classifier,
    model_registry,
)

print(f"Project directory: {PROJECT_DIR}")

## Configuration

The bins mirror the approximate target ranges used when the AI samples were generated. Predictions are cached per model because running the complete test set through every transformer can take a while. The cache is invalidated when the artifact's file sizes or modification times change.

In [ ]:
MODEL_NAMES = (
    "logreg",
    "distilbert",
    "distilbert-lora",
    "distilbert-mica",
    "modernbert",
    "gpt2-variable",
    "gpt2-fixed",
    "qwen3-variable",
    "qwen3-fixed",
)
MODEL_BATCH_SIZES = {
    "logreg": 512,
    "distilbert": 32,
    "distilbert-lora": 32,
    "distilbert-mica": 32,
    "modernbert": 2,
    "gpt2-variable": 8,
    "gpt2-fixed": 8,
    "qwen3-variable": 2,
    "qwen3-fixed": 2,
}
DEVICE = "auto"
INFERENCE_CHUNK_SIZE = 512
REUSE_CACHED_PREDICTIONS = True

LENGTH_BIN_EDGES = [0, 60, 120, 300, 600, np.inf]
LENGTH_BIN_LABELS = ["≤60", "61–120", "121–300", "301–600", "601+"]

ANALYSIS_DIR = PROJECT_DIR / "scripts" / "12_length-bias"
OUTPUT_DIR = ANALYSIS_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Check the model exports

Unavailable models will be reported explicitly and omitted from the current run. After their notebook export cells have created the missing weights, rerunning this notebook will include them automatically.

In [ ]:
registry = model_registry(PROJECT_DIR)
status_rows = []
ready_model_names = []
for model_name in MODEL_NAMES:
    spec = registry[model_name]
    ready, status = artifact_status(spec)
    status_rows.append(
        {
            "model": model_name,
            "ready": ready,
            "status": status,
            "artifact": str(spec.artifact_path),
        }
    )
    if ready:
        ready_model_names.append(model_name)

model_status = pd.DataFrame(status_rows).set_index("model")
if not ready_model_names:
    raise RuntimeError("No complete classifier exports are available")

print("Ready models:", ", ".join(ready_model_names))
missing_models = [
    name for name in MODEL_NAMES if name not in ready_model_names
]
if missing_models:
    print("Unavailable models:", ", ".join(missing_models))

model_status

## Load the held-out test set

The original train and validation splits are not used in this analysis. Each length bin should contain both human and AI examples; otherwise class-specific rates for that bin would be undefined.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rasbt/human-vs-ai-50k")
test_dataset = dataset["test"]

base_frame = pd.DataFrame(
    {
        "id": [str(value) for value in test_dataset["id"]],
        "label": np.asarray(test_dataset["label"], dtype=np.int64),
        "word_count": np.asarray(
            test_dataset["word_count"], dtype=np.int64
        ),
    }
)
base_frame["length_bin"] = pd.cut(
    base_frame["word_count"],
    bins=LENGTH_BIN_EDGES,
    labels=LENGTH_BIN_LABELS,
    include_lowest=True,
    right=True,
)
if base_frame["length_bin"].isna().any():
    raise ValueError("At least one test sample did not fit a length bin")

length_distribution = (
    base_frame.groupby(["length_bin", "label"], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "human", 1: "ai"})
)
if (length_distribution[["human", "ai"]] == 0).any().any():
    raise ValueError("Every length bin must contain both labels")

length_distribution

## Compute calibrated test-set scores

Each exported classifier already includes its fitted calibration parameters. This stage stores the AI probability and the 0.5-threshold prediction for every test example.

In [ ]:
def artifact_signature(spec):
    path = spec.artifact_path
    files = [path] if path.is_file() else sorted(
        file for file in path.rglob("*") if file.is_file()
    )
    digest = hashlib.sha256()
    for file in files:
        stat = file.stat()
        relative_name = file.name if path.is_file() else str(
            file.relative_to(path)
        )
        digest.update(relative_name.encode("utf-8"))
        digest.update(str(stat.st_size).encode("ascii"))
        digest.update(str(stat.st_mtime_ns).encode("ascii"))
    return digest.hexdigest()


def cached_predictions(
    cache_path, expected_ids, signature
):
    if not REUSE_CACHED_PREDICTIONS or not cache_path.is_file():
        return None
    cached = pd.read_csv(cache_path, dtype={"id": str})
    cached_ids = cached["id"].tolist()
    if cached_ids != expected_ids:
        raise ValueError(f"Cached sample IDs do not match: {cache_path}")
    signatures = cached["artifact_signature"].dropna().unique()
    if len(signatures) != 1 or signatures[0] != signature:
        print(f"Ignoring stale cache: {cache_path.name}")
        return None
    print(f"Using cached predictions: {cache_path.name}")
    return cached


In [ ]:
expected_ids = base_frame["id"].tolist()
test_texts = [str(text) for text in test_dataset["text"]]
prediction_frames = []
runtime_rows = []

for model_name in ready_model_names:
    spec = registry[model_name]
    signature = artifact_signature(spec)
    cache_path = OUTPUT_DIR / f"{model_name}-test-predictions.csv"
    predictions = cached_predictions(
        cache_path, expected_ids, signature
    )

    if predictions is None:
        classifier = load_classifier(model_name, device=DEVICE)
        batch_size = MODEL_BATCH_SIZES[model_name]
        probabilities = []
        start_time = time.perf_counter()
        for start in range(0, len(test_texts), INFERENCE_CHUNK_SIZE):
            stop = min(start + INFERENCE_CHUNK_SIZE, len(test_texts))
            probabilities.extend(
                classifier.score_many(
                    test_texts[start:stop], batch_size=batch_size
                )
            )
            print(
                f"{model_name}: {stop:,}/{len(test_texts):,}"
            )
        runtime_seconds = time.perf_counter() - start_time

        predictions = base_frame.copy()
        predictions["model"] = model_name
        predictions["ai_probability"] = np.asarray(
            probabilities, dtype=np.float64
        )
        predictions["prediction"] = (
            predictions["ai_probability"] >= 0.5
        ).astype(np.int64)
        predictions["artifact_signature"] = signature
        predictions.to_csv(cache_path, index=False)

        del classifier
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()
        except ImportError:
            pass
    else:
        runtime_seconds = np.nan

    prediction_frames.append(predictions)
    runtime_rows.append(
        {"model": model_name, "runtime_seconds": runtime_seconds}
    )

all_predictions = pd.concat(prediction_frames, ignore_index=True)
inference_runtimes = pd.DataFrame(runtime_rows).set_index("model")
inference_runtimes

## Summarize scores and errors by length

The mean scores show the direction of any score drift. Score separation is the difference between the mean AI score for actual AI and actual human text. False-positive rate is computed among human texts, while false-negative rate is computed among AI texts.

In [ ]:
def safe_rate(numerator, denominator):
    return numerator / denominator if denominator else np.nan


summary_rows = []
for model_name in ready_model_names:
    model_frame = all_predictions[
        all_predictions["model"] == model_name
    ]
    for length_bin in LENGTH_BIN_LABELS:
        group = model_frame[
            model_frame["length_bin"] == length_bin
        ]
        human = group[group["label"] == 0]
        ai = group[group["label"] == 1]
        true_negative = int((human["prediction"] == 0).sum())
        false_positive = int((human["prediction"] == 1).sum())
        false_negative = int((ai["prediction"] == 0).sum())
        true_positive = int((ai["prediction"] == 1).sum())
        mean_human_score = human["ai_probability"].mean() * 100
        mean_ai_score = ai["ai_probability"].mean() * 100
        summary_rows.append(
            {
                "model": model_name,
                "length_bin": length_bin,
                "samples": len(group),
                "human_samples": len(human),
                "ai_samples": len(ai),
                "accuracy": (group["label"] == group["prediction"]).mean(),
                "mean_human_score": mean_human_score,
                "mean_ai_score": mean_ai_score,
                "score_separation": mean_ai_score - mean_human_score,
                "false_positive_rate": safe_rate(
                    false_positive, false_positive + true_negative
                ),
                "false_negative_rate": safe_rate(
                    false_negative, false_negative + true_positive
                ),
                "brier_score": np.mean(
                    (group["ai_probability"] - group["label"]) ** 2
                ),
            }
        )

length_summary = pd.DataFrame(summary_rows)
length_summary.to_csv(
    OUTPUT_DIR / "length-bin-summary.csv", index=False
)
length_summary.style.format(
    {
        "accuracy": "{:.2%}",
        "mean_human_score": "{:.2f}",
        "mean_ai_score": "{:.2f}",
        "score_separation": "{:.2f}",
        "false_positive_rate": "{:.2%}",
        "false_negative_rate": "{:.2%}",
        "brier_score": "{:.4f}",
    }
)

In [ ]:
def rank_correlation(frame):
    if len(frame) < 2 or frame["word_count"].nunique() < 2:
        return np.nan
    return frame["word_count"].rank().corr(
        frame["ai_probability"].rank()
    )


model_rows = []
for model_name in ready_model_names:
    model_bins = length_summary[
        length_summary["model"] == model_name
    ].set_index("length_bin").loc[LENGTH_BIN_LABELS]
    model_predictions = all_predictions[
        all_predictions["model"] == model_name
    ]
    human_predictions = model_predictions[
        model_predictions["label"] == 0
    ]
    ai_predictions = model_predictions[
        model_predictions["label"] == 1
    ]
    first_bin = model_bins.iloc[0]
    last_bin = model_bins.iloc[-1]
    model_rows.append(
        {
            "model": model_name,
            "human_score_shift": (
                last_bin["mean_human_score"]
                - first_bin["mean_human_score"]
            ),
            "ai_score_shift": (
                last_bin["mean_ai_score"]
                - first_bin["mean_ai_score"]
            ),
            "separation_change": (
                last_bin["score_separation"]
                - first_bin["score_separation"]
            ),
            "accuracy_range": (
                model_bins["accuracy"].max()
                - model_bins["accuracy"].min()
            ),
            "false_positive_rate_range": (
                model_bins["false_positive_rate"].max()
                - model_bins["false_positive_rate"].min()
            ),
            "false_negative_rate_range": (
                model_bins["false_negative_rate"].max()
                - model_bins["false_negative_rate"].min()
            ),
            "human_length_correlation": rank_correlation(
                human_predictions
            ),
            "ai_length_correlation": rank_correlation(
                ai_predictions
            ),
        }
    )

model_summary = pd.DataFrame(model_rows).set_index("model")
model_summary.to_csv(OUTPUT_DIR / "model-length-bias-summary.csv")
model_summary.style.format(
    {
        "human_score_shift": "{:+.2f}",
        "ai_score_shift": "{:+.2f}",
        "separation_change": "{:+.2f}",
        "accuracy_range": "{:.2%}",
        "false_positive_rate_range": "{:.2%}",
        "false_negative_rate_range": "{:.2%}",
        "human_length_correlation": "{:+.3f}",
        "ai_length_correlation": "{:+.3f}",
    }
)

## Score drift

Each panel uses the full 0–100 score range. Greater vertical separation between actual human and AI texts means the model distinguishes the classes more confidently at that length.

In [ ]:
def spread_pair(
    first, second,
    minimum_gap = 5.0, upper_bound = 100.0,
):
    if abs(first - second) >= minimum_gap:
        return first, second
    midpoint = (first + second) / 2
    return (
        max(0.0, midpoint - minimum_gap / 2),
        min(upper_bound, midpoint + minimum_gap / 2),
    )


plot_models = ready_model_names
column_count = min(3, len(plot_models))
row_count = math.ceil(len(plot_models) / column_count)
figure_height = 3.0 * row_count + 0.9
subtitle_y = 1.0 - 0.48 / figure_height
content_top = 1.0 - 0.78 / figure_height
fig, axes = plt.subplots(
    row_count, column_count,
    figsize=(4.0 * column_count, figure_height),
    sharex=True, sharey=True, squeeze=False,
)
x_values = np.arange(len(LENGTH_BIN_LABELS))
human_color = "#777777"
ai_color = "#1f4e79"

for ax, model_name in zip(axes.ravel(), plot_models):
    model_bins = length_summary[
        length_summary["model"] == model_name
    ].set_index("length_bin").loc[LENGTH_BIN_LABELS]
    human_scores = model_bins["mean_human_score"].to_numpy()
    ai_scores = model_bins["mean_ai_score"].to_numpy()
    ax.plot(
        x_values, human_scores, color=human_color,
        linewidth=1.6, marker="o", markersize=3,
    )
    ax.plot(
        x_values, ai_scores, color=ai_color,
        linewidth=1.8, marker="o", markersize=3,
    )
    human_label_y, ai_label_y = spread_pair(
        human_scores[-1], ai_scores[-1]
    )
    ax.text(
        x_values[-1] + 0.15, human_label_y, "Human",
        color=human_color, fontsize=8, va="center",
    )
    ax.text(
        x_values[-1] + 0.15, ai_label_y, "AI",
        color=ai_color, fontsize=8, va="center",
    )
    ax.set_title(model_name, loc="left", fontsize=10)
    ax.set_xlim(-0.1, x_values[-1] + 0.9)
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 50, 100])
    ax.set_xticks(x_values, LENGTH_BIN_LABELS, rotation=30, ha="right")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#aaaaaa")
    ax.tick_params(color="#aaaaaa", labelsize=8)

for ax in axes.ravel()[len(plot_models):]:
    ax.set_visible(False)

fig.suptitle(
    "Score drift reveals whether length shifts both classes",
    x=0.06, y=0.98, ha="left", fontsize=14,
)
fig.text(
    0.06, subtitle_y,
    "Mean calibrated AI score by true label and word-count range",
    ha="left", color="#555555", fontsize=9,
)
fig.supxlabel("Text length (words)", fontsize=10)
fig.supylabel("Mean AI score", fontsize=10)
fig.tight_layout(rect=[0.04, 0.04, 1, content_top])
SCORE_FIGURE_PATH = ANALYSIS_DIR / "length-bias-scores.svg"
fig.savefig(SCORE_FIGURE_PATH, bbox_inches="tight")
plt.show()

## Classification errors

False positives are human texts classified as AI. False negatives are AI texts classified as human. All panels start at zero so small error-rate differences are not visually exaggerated.

In [ ]:
maximum_error_rate = (
    length_summary[["false_positive_rate", "false_negative_rate"]]
    .max()
    .max()
    * 100
)
error_axis_max = min(100.0, max(5.0, math.ceil(maximum_error_rate / 5) * 5))
fig, axes = plt.subplots(
    row_count, column_count,
    figsize=(4.0 * column_count, figure_height),
    sharex=True, sharey=True, squeeze=False,
)
false_positive_color = "#a6533c"
false_negative_color = "#1f4e79"

for ax, model_name in zip(axes.ravel(), plot_models):
    model_bins = length_summary[
        length_summary["model"] == model_name
    ].set_index("length_bin").loc[LENGTH_BIN_LABELS]
    false_positive_rates = (
        model_bins["false_positive_rate"].to_numpy() * 100
    )
    false_negative_rates = (
        model_bins["false_negative_rate"].to_numpy() * 100
    )
    ax.plot(
        x_values, false_positive_rates, color=false_positive_color,
        linewidth=1.6, marker="o", markersize=3,
    )
    ax.plot(
        x_values, false_negative_rates, color=false_negative_color,
        linewidth=1.6, marker="o", markersize=3,
    )
    fp_label_y, fn_label_y = spread_pair(
        false_positive_rates[-1], false_negative_rates[-1],
        minimum_gap=max(1.0, error_axis_max * 0.08),
        upper_bound=error_axis_max,
    )
    ax.text(
        x_values[-1] + 0.15, fp_label_y, "FPR",
        color=false_positive_color, fontsize=8, va="center",
    )
    ax.text(
        x_values[-1] + 0.15, fn_label_y, "FNR",
        color=false_negative_color, fontsize=8, va="center",
    )
    ax.set_title(model_name, loc="left", fontsize=10)
    ax.set_xlim(-0.1, x_values[-1] + 0.9)
    ax.set_ylim(0, error_axis_max)
    ax.set_xticks(x_values, LENGTH_BIN_LABELS, rotation=30, ha="right")
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#aaaaaa")
    ax.tick_params(color="#aaaaaa", labelsize=8)

for ax in axes.ravel()[len(plot_models):]:
    ax.set_visible(False)

fig.suptitle(
    "Length-dependent errors reveal class-specific failures",
    x=0.06, y=0.98, ha="left", fontsize=14,
)
fig.text(
    0.06, subtitle_y,
    "False-positive and false-negative rates by word-count range",
    ha="left", color="#555555", fontsize=9,
)
fig.supxlabel("Text length (words)", fontsize=10)
fig.supylabel("Error rate (%)", fontsize=10)
fig.tight_layout(rect=[0.04, 0.04, 1, content_top])
ERROR_FIGURE_PATH = ANALYSIS_DIR / "length-bias-error-rates.svg"
fig.savefig(ERROR_FIGURE_PATH, bbox_inches="tight")
plt.show()

## How to interpret the summary

- `human_score_shift` and `ai_score_shift` compare the longest bin with the shortest bin. Same-sign shifts suggest that length moves both classes in one direction.
- `separation_change` measures whether the average score gap between AI and human examples widens with length. A positive value indicates better class separation for long texts.
- The two rank correlations capture monotonic score changes without assuming that the relationship is linear.
- The error-rate ranges show whether one class is affected more strongly than the other.

A length association can still reflect topic, source, or generator differences. A stronger follow-up would compare human and AI examples matched by source and target length, or fit a regression that controls for these variables.